In [4]:
import requests

headers = {
    'sec-ch-ua-platform': '"macOS"',
    'Referer': 'https://www.atptour.com/en/tournaments',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'sec-ch-ua': '"Google Chrome";v="153", "Not_A Brand";v="8", "Chromium";v="153"',
    'sec-ch-ua-mobile': '?0',
}

response = requests.get('https://www.atptour.com/en/-/tournaments/calendar/tour', headers=headers, verify=False)
response_json = response.json()

/Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.atptour.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [9]:
response_json

{'TournamentDates': [{'DisplayDate': 'January, 2026',
   'IsExpanded': False,
   'NoEvents': 6,
   'Tournaments': [{'Id': '9900',
     'Name': 'United Cup',
     'Location': 'Perth-Sydney, Australia',
     'FormattedDate': '2 - 11 January, 2026',
     'IsLive': False,
     'IsPastEvent': True,
     'ScoresUrl': '/en/scores/archive/perth-sydney/9900/2026/country-results',
     'DrawsUrl': '/en/scores/archive/perth-sydney/9900/2026/country-draws',
     'TournamentSiteUrl': 'https://www.unitedcup.com/en/',
     'ScheduleUrl': '/en/scores/current/united-cup/9900/country-schedule',
     'Type': 'UC',
     'SinglesDrawPrintUrl': 'http://www.protennislive.com/posting/2026/9900/mds.pdf',
     'DoublesDrawPrintUrl': 'http://www.protennislive.com/posting/2026/9900/mdd.pdf',
     'QualySinglesDrawPrintUrl': 'http://www.protennislive.com/posting/2026/9900/qs.pdf',
     'SchedulePrintUrl': 'http://www.protennislive.com/posting/2026/9900/op.pdf',
     'CountryFlagUrl': '/-/media/images/flags/aus.svg

## Clean up the tournament calendar response

`response_json["TournamentDates"]` is a list of month buckets, each holding a
`Tournaments` list. Flatten those into one table, parse `FormattedDate` into
real `start_date`/`end_date` values, and write the result out.

Note: this ATP calendar endpoint only returns the *current* tour year (2026
here) - it has no historical/2025 data, so it's only useful going forward.

In [6]:
import calendar
import re
from datetime import date

MONTH_NUMBERS = {name: num for num, name in enumerate(calendar.month_name) if name}


def parse_formatted_date(formatted_date: str) -> tuple[date, date]:
    """Parse e.g. "18 January - 1 February, 2026" / "9 - 13 December, 2026" into (start, end)."""
    match = re.match(r"^(.*?)\s*-\s*(\d{1,2}\s+\w+),\s*(\d{4})$", formatted_date.strip())
    if not match:
        raise ValueError(f"Unrecognized FormattedDate: {formatted_date!r}")
    start_part, end_part, year_str = match.groups()
    year = int(year_str)

    end_day_str, end_month_str = end_part.split()
    end_date = date(year, MONTH_NUMBERS[end_month_str], int(end_day_str))

    start_tokens = start_part.split()
    if len(start_tokens) == 1:
        start_date = date(year, end_date.month, int(start_tokens[0]))
    else:
        start_day_str, start_month_str = start_tokens
        start_date = date(year, MONTH_NUMBERS[start_month_str], int(start_day_str))

    return start_date, end_date


In [7]:
import pandas as pd

raw_tournaments = [
    tournament
    for month_block in response_json["TournamentDates"]
    for tournament in month_block["Tournaments"]
]

COLUMN_RENAME = {
    "Id": "tournament_id",
    "Name": "tournament_name",
    "Location": "location",
    "Type": "tour_level",
    "EventType": "event_type",
    "Surface": "surface",
    "IndoorOutdoor": "indoor_outdoor",
    "SglDrawSize": "sgl_draw_size",
    "DblDrawSize": "dbl_draw_size",
    "FormattedDate": "formatted_date",
}

df_tournaments = (
    pd.DataFrame(raw_tournaments)[list(COLUMN_RENAME)].rename(columns=COLUMN_RENAME)
)
df_tournaments["surface"] = df_tournaments["surface"].str.strip().str.lower().replace("", pd.NA)
df_tournaments[["city", "country"]] = df_tournaments["location"].str.split(", ", n=1, expand=True)

start_end = df_tournaments["formatted_date"].map(parse_formatted_date)
df_tournaments["start_date"] = start_end.map(lambda pair: pair[0])
df_tournaments["end_date"] = start_end.map(lambda pair: pair[1])
df_tournaments["year"] = df_tournaments["start_date"].map(lambda d: d.year)
df_tournaments["tour"] = "atp"

df_tournaments = df_tournaments.drop(columns=["formatted_date"]).sort_values("start_date").reset_index(drop=True)
df_tournaments


,tournament_id,tournament_name,location,tour_level,event_type,surface,indoor_outdoor,sgl_draw_size,dbl_draw_size,city,country,start_date,end_date,year,tour
0,9900,United Cup,"Perth-Sydney, Australia",UC,Tour,hard,Outdoor,18,18,Perth-Sydney,Australia,2026-01-02,2026-01-11,2026,atp
1,339,Brisbane International presented by ANZ,"Brisbane, Australia",250,Tour,hard,Outdoor,32,24,Brisbane,Australia,2026-01-04,2026-01-11,2026,atp
2,336,Bank of China Hong Kong Tennis Open,"Hong Kong, Hong Kong",250,Tour,hard,Outdoor,28,16,Hong Kong,Hong Kong,2026-01-05,2026-01-11,2026,atp
3,8998,Adelaide International,"Adelaide, Australia",250,Tour,hard,Outdoor,28,24,Adelaide,Australia,2026-01-12,2026-01-17,2026,atp
4,301,ASB Classic,"Auckland, New Zealand",250,Tour,hard,Outdoor,28,16,Auckland,New Zealand,2026-01-12,2026-01-17,2026,atp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,352,Rolex Paris Masters,"Paris, France",1000,Tour,hard,Indoor,56,24,Paris,France,2026-11-02,2026-11-08,2026,atp
61,429,Bybit Stockholm Open,"Stockholm, Sweden",250,Tour,hard,Indoor,28,16,Stockholm,Sweden,2026-11-08,2026-11-14,2026,atp
62,605,Nitto ATP Finals,"Turin, Italy",WC,Tour,hard,Indoor,8,0,Turin,Italy,2026-11-15,2026-11-22,2026,atp
63,8099,Davis Cup Finals,"Bologna, Italy",DCR,Other,hard,Indoor,2,1,Bologna,Italy,2026-11-23,2026-11-29,2026,atp


In [8]:
YEAR = df_tournaments["year"].iloc[0]

df_tournaments.to_csv(f"atp_tournaments_{YEAR}.csv", index=False)
df_tournaments.to_json(f"atp_tournaments_{YEAR}.json", orient="records", date_format="iso", indent=2)


## Bonus: tournament ids are permanent, and there's a 2025 schedule table too

The numeric id in a tournament URL (`/en/tournaments/{slug}/{id}/overview`,
e.g. Delray Beach = 499, Montpellier = 375) is a **permanent per-event id**,
not tied to the current season - the same ids show up in ATP's
["What is the 2025 ATP Tour calendar"](https://www.atptour.com/en/news/what-is-the-2025-atp-tour-calendar)
news article. That article is even more useful than the id links alone: it
has a full HTML table of the 2025 season (dates, surface, level) for every
tournament, which solves the "API only has 2026" limitation from above.

No `bs4`/`lxml` in this project's env, so the table below is parsed with
plain `re` against the fairly regular `<tr><td>...</td>...</tr>` markup.

In [10]:
import subprocess
from pathlib import Path

CALENDAR_YEAR = 2025
CALENDAR_ARTICLE_URL = "https://www.atptour.com/en/news/what-is-the-2025-atp-tour-calendar"
CALENDAR_HTML_PATH = Path(f"atp_{CALENDAR_YEAR}_calendar_article.html")

# `requests` hangs indefinitely against this article URL from this env (curl is fine) - shell
# out to curl instead, and cache the result so this only needs to succeed once.
if not CALENDAR_HTML_PATH.exists():
    subprocess.run(
        ["curl", "-sL", "-A", headers["User-Agent"], "-o", str(CALENDAR_HTML_PATH), CALENDAR_ARTICLE_URL],
        check=True,
        timeout=30,
    )
calendar_html = CALENDAR_HTML_PATH.read_text()
len(calendar_html)

115121

In [14]:
ROW_PATTERN = re.compile(r"<tr>\s*(.*?)\s*</tr>", re.DOTALL)
CELL_PATTERN = re.compile(r"<td>(.*?)</td>", re.DOTALL)
ANCHOR_PATTERN = re.compile(r"<a[^>]*href=['\"]([^'\"]+)['\"][^>]*>(.*?)</a>", re.DOTALL)
TOURNAMENT_URL_PATTERN = re.compile(r"/tournaments/([a-z0-9-]+)/(\d+)/overview")


def strip_tags(html_fragment: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html_fragment).replace("&nbsp;", " ").replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def parse_calendar_row(row_html: str) -> dict | None:
    cells = CELL_PATTERN.findall(row_html)
    if len(cells) != 4 or not re.match(r"^\s*\d", cells[0]):
        return None  # skip the header row (and anything malformed)
    date_range_cell, name_cell, surface_cell, level_cell = cells

    tournament_id = tournament_slug = None
    anchor_match = ANCHOR_PATTERN.search(name_cell)
    if anchor_match:
        href, anchor_text = anchor_match.groups()
        url_match = TOURNAMENT_URL_PATTERN.search(href)
        if url_match:
            tournament_slug, tournament_id = url_match.groups()
        name_cell = name_cell[: anchor_match.start()] + anchor_text + name_cell[anchor_match.end() :]

    name_cell = name_cell.replace("<br>", "\n").replace("<br/>", "\n").replace("&nbsp;", " ").replace("\xa0", " ")
    lines = [line.strip() for line in re.sub(r"<[^>]+>", "", name_cell).split("\n") if line.strip()]

    return {
        "date_range": strip_tags(date_range_cell),
        "tournament_name": lines[0] if lines else None,
        "location": lines[1] if len(lines) > 1 else None,
        "surface": strip_tags(surface_cell),
        "tour_level": strip_tags(level_cell),
        "tournament_id": tournament_id,
        "tournament_slug": tournament_slug,
    }


calendar_rows_2025 = [
    row for row_html in ROW_PATTERN.findall(calendar_html) if (row := parse_calendar_row(row_html))
]
print(f"Parsed {len(calendar_rows_2025)} tournament rows")
print(f"Rows missing a resolvable tournament_id: {sum(r['tournament_id'] is None for r in calendar_rows_2025)}")

Parsed 63 tournament rows
Rows missing a resolvable tournament_id: 13


In [15]:
from datetime import datetime


def _parse_day_month_year(day: str, month: str, year: int) -> date:
    for month_format in ("%b", "%B"):  # the source mixes abbreviated ("Jul") and full ("July") names
        try:
            return datetime.strptime(f"{day} {month} {year}", f"%d {month_format} %Y").date()
        except ValueError:
            continue
    raise ValueError(f"Unrecognized month name: {month!r}")


def parse_calendar_date_range(date_range: str, year: int) -> tuple[date, date]:
    """Parse e.g. "27 Jan-2 Feb" / "9 Nov-16 Nov" into (start, end); handles a Dec->Jan rollover."""
    match = re.match(r"^(\d{1,2})\s+(\w+)-(\d{1,2})\s+(\w+)$", date_range.strip())
    if not match:
        raise ValueError(f"Unrecognized calendar date range: {date_range!r}")
    start_day, start_mon, end_day, end_mon = match.groups()

    end_date = _parse_day_month_year(end_day, end_mon, year)
    start_date = _parse_day_month_year(start_day, start_mon, year)
    if start_date > end_date:  # e.g. United Cup: "27 Dec-5 Jan" actually starts the prior year
        start_date = start_date.replace(year=year - 1)

    return start_date, end_date


df_tournaments_2025 = pd.DataFrame(calendar_rows_2025)
df_tournaments_2025["surface"] = df_tournaments_2025["surface"].str.lower()
df_tournaments_2025[["city", "country"]] = df_tournaments_2025["location"].str.split(", ", n=1, expand=True)

start_end_2025 = df_tournaments_2025["date_range"].map(lambda dr: parse_calendar_date_range(dr, CALENDAR_YEAR))
df_tournaments_2025["start_date"] = start_end_2025.map(lambda pair: pair[0])
df_tournaments_2025["end_date"] = start_end_2025.map(lambda pair: pair[1])
df_tournaments_2025["year"] = CALENDAR_YEAR
df_tournaments_2025["tour"] = "atp"

df_tournaments_2025 = df_tournaments_2025.drop(columns=["date_range"]).sort_values("start_date").reset_index(drop=True)
df_tournaments_2025

,tournament_name,location,surface,tour_level,tournament_id,tournament_slug,city,country,start_date,end_date,year,tour
0,United Cup,"Perth and Sydney, Australia",hard,United Cup,NaN,NaN,Perth and Sydney,Australia,2024-12-27,2025-01-05,2025,atp
1,Brisbane International presented by Evie,"Brisbane, Australia",hard,ATP 250,NaN,NaN,Brisbane,Australia,2024-12-29,2025-01-05,2025,atp
2,Bank of China Hong Kong Tennis Open,"Hong Kong, Hong Kong",hard,ATP 250,336,hong-kong,Hong Kong,Hong Kong,2024-12-30,2025-01-05,2025,atp
3,Adelaide International,"Adelaide, Australia",hard,ATP 250,8998,adelaide,Adelaide,Australia,2025-01-06,2025-01-11,2025,atp
4,ASB Classic,"Auckland, New Zealand",hard,ATP 250,301,auckland,Auckland,New Zealand,2025-01-06,2025-01-11,2025,atp
...,...,...,...,...,...,...,...,...,...,...,...,...
58,Belgrade Open,"Belgrade, Serbia",hard,ATP 250,NaN,NaN,Belgrade,Serbia,2025-11-02,2025-11-08,2025,atp
59,Moselle Open,"Metz, France",hard,ATP 250,NaN,NaN,Metz,France,2025-11-02,2025-11-08,2025,atp
60,Nitto ATP Finals,"Turin, Italy",hard,Nitto ATP Finals,605,nitto-atp-finals,Turin,Italy,2025-11-09,2025-11-16,2025,atp
61,Davis Cup Finals,"Bologna, Italy",hard,Davis Cup,8099,davis-cup-finals,Bologna,Italy,2025-11-18,2025-11-23,2025,atp


In [16]:
# Confirm the "permanent id" theory: every id present in both years should map to the same slug.
ids_2025 = df_tournaments_2025.dropna(subset=["tournament_id"])
overlap = ids_2025.merge(
    df_tournaments[["tournament_id", "tournament_name"]],
    on="tournament_id",
    how="inner",
    suffixes=("_2025", "_2026"),
)
print(f"{len(ids_2025)}/{len(df_tournaments_2025)} 2025 rows have a direct id")
print(f"{len(overlap)} of those ids also appear in the 2026 data (same permanent id, confirmed)")
overlap[["tournament_id", "tournament_name_2025", "tournament_name_2026"]].head(10)


50/63 2025 rows have a direct id
50 of those ids also appear in the 2026 data (same permanent id, confirmed)


,tournament_id,tournament_name_2025,tournament_name_2026
0,336,Bank of China Hong Kong Tennis Open,Bank of China Hong Kong Tennis Open
1,8998,Adelaide International,Adelaide International
2,301,ASB Classic,ASB Classic
3,580,Australian Open,Australian Open
4,375,Open Occitanie,Open Occitanie
5,8096,Davis Cup Qualifiers 1st Rd,Davis Cup Qualifiers 1st Rd
6,407,ABN AMRO Open,ABN AMRO Open
7,499,Delray Beach Open,Delray Beach Open
8,506,IEB+ Argentina Open,IEB+ Argentina Open
9,451,Qatar ExxonMobil Open,Qatar ExxonMobil Open


In [17]:
from difflib import SequenceMatcher

# Fill the remaining ~13 gaps: the article's own links miss some rows (e.g. United Cup, Brisbane),
# but the permanent id is still discoverable via the 2026 reference table since ids don't change.
missing_mask = df_tournaments_2025["tournament_id"].isna()


def best_2026_match(name: str, city: str) -> tuple[str | None, float]:
    scores = df_tournaments["tournament_name"].map(lambda n: SequenceMatcher(None, name, n).ratio())
    city_scores = df_tournaments["city"].map(lambda c: SequenceMatcher(None, city, c).ratio())
    combined = 0.6 * scores + 0.4 * city_scores
    best_idx = combined.idxmax()
    return df_tournaments.loc[best_idx, "tournament_id"], combined.loc[best_idx]


matches = df_tournaments_2025.loc[missing_mask].apply(
    lambda row: best_2026_match(row["tournament_name"], row["city"]), axis=1
)
df_tournaments_2025.loc[missing_mask, "tournament_id"] = matches.map(lambda m: m[0])
df_tournaments_2025.loc[missing_mask, "match_score"] = matches.map(lambda m: m[1])

df_tournaments_2025.loc[missing_mask, ["tournament_name", "city", "tournament_id", "match_score"]]


,tournament_name,city,tournament_id,match_score
0,United Cup,Perth and Sydney,9900,0.914286
1,Brisbane International presented by Evie,Brisbane,339,0.946835
8,Dallas Open,Dallas,424,0.888889
10,Open 13 Provence,Marseille,375,0.440000
15,Movistar Chile Open,Santiago,8996,0.780488
33,Mallorca Championships presented by Ecotrans G...,Mallorca,8994,0.680851
41,Mubadala Citi DC Open,Washington,418,0.918919
42,National Bank Open Presented by Rogers,Toronto,6932,0.501978
53,BNP Paribas Nordic Open,Stockholm,429,0.651163
54,European Open,Brussels,7485,0.746667


In [18]:
# A few of those fuzzy guesses are wrong (e.g. Toronto's National Bank Open matched to Rio de
# Janeiro, Open 13 Provence matched to Montpellier) - both scored low, so drop anything under a
# confidence bar rather than keep a wrong id. Left blank for manual completion.
MIN_FUZZY_MATCH_SCORE = 0.65
unreliable = missing_mask & (df_tournaments_2025["match_score"] < MIN_FUZZY_MATCH_SCORE)
df_tournaments_2025.loc[unreliable, ["tournament_id", "match_score"]] = pd.NA

print(f"{df_tournaments_2025['tournament_id'].notna().sum()}/{len(df_tournaments_2025)} rows now have a tournament_id")
df_tournaments_2025.loc[df_tournaments_2025["tournament_id"].isna(), ["tournament_name", "city"]]


58/63 rows now have a tournament_id


,tournament_name,city
10,Open 13 Provence,Marseille
42,National Bank Open Presented by Rogers,Toronto
58,Belgrade Open,Belgrade
59,Moselle Open,Metz
62,Next Gen ATP Finals presented by PIF,Jeddah


In [19]:
df_tournaments_2025 = df_tournaments_2025.drop(columns=["tournament_slug", "match_score"])
df_tournaments_2025.to_csv(f"atp_tournaments_{CALENDAR_YEAR}.csv", index=False)
df_tournaments_2025.to_json(f"atp_tournaments_{CALENDAR_YEAR}.json", orient="records", date_format="iso", indent=2)
